# 04 - Backtest Results

This notebook summarizes the final comparison currently available for Presentation 1. It compares transparent rule-based strategies, DMN-lite, and the first LSTM/Sharpe-loss runs with and without CPD.


In [1]:
import sys
from pathlib import Path

sys.modules.setdefault("numexpr", None)
sys.modules.setdefault("bottleneck", None)

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data" / "processed" / "stoxx600"
PLOT_TEMPLATE = "plotly_white"


## Reproducible Backtest Commands

The backtests are generated by `scripts/04_run_backtest.py`. The optional cell below reruns the model-position backtests and rebuilds the final comparison file.


In [2]:
RUN_BACKTEST = False

if RUN_BACKTEST:
    import subprocess
    from src.backtest import performance_summary

    subprocess.run(
        [
            sys.executable,
            str(PROJECT_ROOT / "scripts" / "04_run_backtest.py"),
            "--positions-file", "dmn_lite_positions.csv",
            "--position-col", "dmn_lite_position",
            "--out-returns", "backtest_returns_with_dmn.csv",
            "--out-summary", "backtest_summary_with_dmn.csv",
        ],
        cwd=PROJECT_ROOT,
        check=True,
    )
    subprocess.run(
        [
            sys.executable,
            str(PROJECT_ROOT / "scripts" / "04_run_backtest.py"),
            "--positions-file", "dmn_lstm_positions.csv",
            "--position-col", "dmn_lstm_position",
            "--out-returns", "backtest_returns_with_lstm.csv",
            "--out-summary", "backtest_summary_with_lstm.csv",
        ],
        cwd=PROJECT_ROOT,
        check=True,
    )
    subprocess.run(
        [
            sys.executable,
            str(PROJECT_ROOT / "scripts" / "04_run_backtest.py"),
            "--positions-file", "dmn_lstm_no_cpd_positions.csv",
            "--position-col", "dmn_lstm_position",
            "--out-returns", "backtest_returns_with_lstm_no_cpd.csv",
            "--out-summary", "backtest_summary_with_lstm_no_cpd.csv",
        ],
        cwd=PROJECT_ROOT,
        check=True,
    )


## Performance Summary

All strategies are evaluated after transaction costs and portfolio-level volatility scaling. Rule-based strategies start after enough lookback history is available; supervised models start in 2010 because they need an initial training window.


In [3]:
summary = pd.read_csv(DATA_DIR / "backtest_summary_final_comparison.csv", parse_dates=["start_date", "end_date"])
order = ["dmn_lite", "slow_momentum", "dmn_lstm_no_cpd", "cpd_adjusted", "dmn_lstm", "slow_fast"]
summary["strategy"] = pd.Categorical(summary["strategy"], categories=order, ordered=True)
summary = summary.sort_values("strategy").reset_index(drop=True)
summary["strategy_label"] = summary["strategy"].map({
    "dmn_lite": "DMN-lite",
    "slow_momentum": "Slow momentum",
    "dmn_lstm_no_cpd": "LSTM DMN without CPD",
    "cpd_adjusted": "CPD-adjusted rule",
    "dmn_lstm": "LSTM DMN with CPD",
    "slow_fast": "Slow + fast",
})

display_cols = [
    "strategy_label", "start_date", "end_date", "ann_return", "ann_vol", "sharpe",
    "sortino", "calmar", "max_drawdown", "hit_ratio", "avg_assets", "avg_turnover",
]
pretty = summary[display_cols].copy()
for col in ["ann_return", "ann_vol", "max_drawdown", "hit_ratio", "avg_turnover"]:
    pretty[col] = pretty[col].map(lambda x: f"{x:.2%}")
for col in ["sharpe", "sortino", "calmar", "avg_assets"]:
    pretty[col] = pretty[col].map(lambda x: f"{x:.2f}")
pretty


,strategy_label,start_date,end_date,ann_return,ann_vol,sharpe,sortino,calmar,max_drawdown,hit_ratio,avg_assets,avg_turnover
0,DMN-lite,2010-01-04,2026-04-10,2.79%,7.45%,0.41,0.51,0.14,-19.31%,52.48%,317.89,3.59%
1,Slow momentum,2006-12-21,2026-04-10,3.75%,14.73%,0.32,0.42,0.11,-34.29%,53.00%,251.34,4.19%
2,LSTM DMN without CPD,2010-01-04,2026-04-10,2.69%,13.59%,0.26,0.31,0.09,-30.85%,53.53%,312.03,9.71%
3,CPD-adjusted rule,2006-12-21,2026-04-10,2.61%,13.24%,0.26,0.33,0.08,-31.35%,53.34%,251.29,5.97%
4,LSTM DMN with CPD,2010-01-04,2026-04-10,2.03%,13.57%,0.22,0.27,0.07,-29.26%,52.75%,312.03,8.47%
5,Slow + fast,2006-12-21,2026-04-10,1.68%,12.91%,0.19,0.24,0.05,-33.68%,53.30%,251.29,4.53%


In [4]:
returns = pd.read_csv(DATA_DIR / "backtest_returns_final_comparison.csv", parse_dates=["date"])
returns["strategy_label"] = returns["strategy"].map(dict(zip(summary["strategy"].astype(str), summary["strategy_label"])))
returns["equity"] = returns.groupby("strategy_label", observed=True)["net_return"].transform(lambda x: (1.0 + x.fillna(0.0)).cumprod())

fig = px.line(
    returns,
    x="date",
    y="equity",
    color="strategy_label",
    title="Cumulative performance after costs",
    labels={"date": "Date", "equity": "Growth of 1", "strategy_label": "Strategy"},
)
fig.update_layout(template=PLOT_TEMPLATE, height=540)
fig.show()


In [5]:
drawdowns = []
for strategy, group in returns.groupby("strategy_label", observed=True):
    equity = group["equity"]
    dd = equity / equity.cummax() - 1.0
    drawdowns.append(pd.DataFrame({"date": group["date"], "strategy_label": strategy, "drawdown": dd}))
drawdowns = pd.concat(drawdowns, ignore_index=True)

fig = px.line(
    drawdowns,
    x="date",
    y="drawdown",
    color="strategy_label",
    title="Strategy drawdowns",
    labels={"date": "Date", "drawdown": "Drawdown", "strategy_label": "Strategy"},
)
fig.update_layout(template=PLOT_TEMPLATE, height=540, yaxis_tickformat=".0%")
fig.show()


## Comparable Metrics

These charts are designed for the report and PowerPoint: they compare risk-adjusted performance, downside risk, return/volatility and turnover.


In [6]:
metric_long = summary.melt(
    id_vars="strategy_label",
    value_vars=["sharpe", "sortino", "calmar"],
    var_name="metric",
    value_name="value",
)
fig = px.bar(
    metric_long,
    x="strategy_label",
    y="value",
    color="metric",
    barmode="group",
    title="Risk-adjusted performance comparison",
    labels={"strategy_label": "Strategy", "value": "Metric value", "metric": "Metric"},
)
fig.update_layout(template=PLOT_TEMPLATE, height=500, xaxis_tickangle=-20)
fig.show()


In [7]:
fig = px.scatter(
    summary,
    x="ann_vol",
    y="ann_return",
    size=summary["max_drawdown"].abs(),
    color="strategy_label",
    hover_data=["sharpe", "sortino", "calmar", "hit_ratio", "avg_turnover"],
    title="Return vs volatility, bubble size = max drawdown",
    labels={"ann_vol": "Annualized volatility", "ann_return": "Annualized return", "strategy_label": "Strategy"},
)
fig.update_layout(template=PLOT_TEMPLATE, height=520, xaxis_tickformat=".1%", yaxis_tickformat=".1%")
fig.show()


In [8]:
risk_turnover = summary[["strategy_label", "max_drawdown", "avg_turnover"]].copy()
risk_turnover["max_drawdown_abs"] = risk_turnover["max_drawdown"].abs()
risk_turnover = risk_turnover.melt(
    id_vars="strategy_label",
    value_vars=["max_drawdown_abs", "avg_turnover"],
    var_name="metric",
    value_name="value",
)
risk_turnover["metric"] = risk_turnover["metric"].map({"max_drawdown_abs": "Max drawdown abs.", "avg_turnover": "Average turnover"})

fig = px.bar(
    risk_turnover,
    x="strategy_label",
    y="value",
    color="metric",
    barmode="group",
    title="Risk and trading intensity",
    labels={"strategy_label": "Strategy", "value": "Value", "metric": "Metric"},
)
fig.update_layout(template=PLOT_TEMPLATE, height=500, xaxis_tickangle=-20, yaxis_tickformat=".1%")
fig.show()


## Main Result To Present

The best current Sharpe ratio is obtained by `DMN-lite`. Slow momentum has the highest annualized return, but with materially higher volatility and drawdown. The first LSTM/Sharpe-loss implementation is working end-to-end, yet it is not tuned enough to beat the simpler DMN-lite baseline.

The CPD interpretation should be careful. The rule-based CPD adjustment reduces drawdown versus slow momentum, but it also lowers return and Sharpe. In the first LSTM experiment, the model without CPD outperforms the model with CPD. This suggests that CPD is not automatically valuable as a raw feature; it likely needs better transformations, persistence features, or stronger regularization.


## Current Reading

For Presentation 1, the strongest message is that the project now has a complete, reproducible and walk-forward research pipeline:

1. raw prices and static universe reference;
2. long-format feature panel;
3. idiosyncratic CPD scores;
4. supervised allocation models;
5. backtests with costs, volatility scaling and comparable metrics.

The empirical conclusion is preliminary but useful: a simple learned allocation layer currently gives the cleanest risk-adjusted profile, while the LSTM/Sharpe-loss implementation provides the required bridge toward the paper and the next tuning problem.
